# Transfer Learning for Toxicity Prediction: Addressing Data Scarcity in Environmental Risk Assessment
## SOT 2027 Continuing Education · MNP tutorial
Click **Runtime → Run all**. CPU is sufficient; no Drive mount, uploads or repository clone is required. Setup runs first, followed by the five slide-aligned sections.

Based on [Chen et al. (2025), JHM 139353](https://doi.org/10.1016/j.jhazmat.2025.139353) and the [original research repository](https://github.com/choulab210/Micro-Nanoplastics).
This simplified demonstration teaches parameter transfer, not independent external validation. Class 1 means the CSV label `Negative`, whose biological interpretation is unresolved.

## Part 0. Environment Setup and Data Preparation (Run Once)

Run this section once to install the required Python packages, download the MNP dataset, and prepare the supporting functions used throughout this tutorial. No nanoparticle source-model pretraining is performed.

### 0.1 Install Required Packages

PyTorch defines and trains the neural network; NumPy and pandas handle numerical data and tables. Scikit-learn supplies the Random Forest and evaluation metrics, and matplotlib draws the figures. The existing version checks install the tested package versions when needed.

In [ ]:
# Install tested versions before importing scientific packages.
import sys, subprocess, importlib.metadata
required = {'torch':'2.13.0', 'numpy':'2.4.6', 'pandas':'2.3.3',
            'scikit-learn':'1.9.0', 'matplotlib':'3.11.1'}
missing = []
for package, version in required.items():
    try:
        present = importlib.metadata.version(package).split('+')[0]
    except importlib.metadata.PackageNotFoundError:
        present = None
    if present != version:
        missing.append(f'{package}=={version}')
if missing:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--quiet', *missing])
print('Dependencies ready; all computations below use CPU.')

### 0.2 Load Supporting Functions

These functions handle data preprocessing, model evaluation, and figure generation. Run the cell once; its code is hidden by default using Colab form view. Use the cell’s Show code control if you want to inspect it; other notebook viewers may display it differently.

In [ ]:
#@title Supporting functions (run once; show code to inspect)
# Supporting helpers are included in this notebook: no repository clone needed.
from pathlib import Path
import urllib.request, urllib.parse, uuid, json, hashlib, platform
import numpy as np
import pandas as pd
import torch
from torch import nn
from IPython.display import display, Image
RUN = Path.cwd() / 'colab_runs' / uuid.uuid4().hex[:12]
RUN.mkdir(parents=True, exist_ok=False)
FIG = RUN / 'figures'
FIG.mkdir()
COMMIT = '6f786a275d1598bd5a7e3bb61c125529744e7dc4'
BASE = f'https://raw.githubusercontent.com/choulab210/Micro-Nanoplastics/{COMMIT}/'

def download_verified(relative_path, destination, expected_sha256):
    request = urllib.request.Request(BASE + urllib.parse.quote(relative_path),
                                     headers={'User-Agent':'SOT-2027-CE-Tutorial'})
    with urllib.request.urlopen(request, timeout=120) as response:
        content = response.read()
    actual = hashlib.sha256(content).hexdigest()
    if actual != expected_sha256:
        raise ValueError(f'Checksum mismatch for {relative_path}: {actual}')
    with destination.open('xb') as file:
        file.write(content)
    print(f'Verified {destination.name}: {actual}')
    return destination

"""Historical N=100 sampling and pre-split scaling; NOT leakage-free evaluation."""
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

FEATURES = ['Coating','Shape','Dose','Size','Exposure Time','HOMO Energy','LUMO Energy']
# Published notebook constants; comments say eV, but unit provenance is unverified.
ORBITALS = {
    'PE': (-0.30393261512793585, 0.11592184908302908),
    'PP': (-0.2938916146961782, 0.10272616946283625),
    'PVC': (-0.2886950926400236, 0.002629107190461072),
    'PS': (-0.23323174962813908, 0.0022633605291856283),
    'PET': (-0.23589308197284242, -0.08382145000478655)}

def prepare_data(path):
    raw = pd.read_csv(path)
    raw['_source_row'] = np.arange(len(raw))
    required = ['Polymer type','Surface functionalization','Size (nm)',
                'Dose (μg/mL)','Shape','Exposure time (h)','Class']
    raw = raw.dropna(subset=required)
    if not set(raw['Class']).issubset({'Negative','Positive'}):
        raise ValueError('Unknown label')
    # Polymer concatenation order matters to the seed-42 sample.
    other = pd.concat([raw[raw['Polymer type']==p] for p in ['PET','PVC','PP']])
    pspe = pd.concat([raw[raw['Polymer type']==p] for p in ['PS','PE']])
    def sample(group, label, n):
        return group[group['Class']==label].sample(n=n, random_state=42)
    ds = pd.concat([sample(pspe,'Negative',33), sample(pspe,'Positive',35),
                    sample(other,'Negative',27), sample(other,'Positive',5)])
    ds = ds.sample(frac=1, random_state=42).reset_index(drop=True)
    if (ds[['Size (nm)','Dose (μg/mL)']].astype(float) <= 0).any().any():
        raise ValueError('Log transforms require positive values')
    x = pd.DataFrame({'Coating': (ds['Surface functionalization']!='Pristine').astype(int),
                      'Shape': (ds['Shape']!='Spherical').astype(int)})
    x['Dose'] = np.log10(ds['Dose (μg/mL)'].astype(float))
    x['Size'] = np.log10(ds['Size (nm)'].astype(float))
    x['Exposure Time'] = ds['Exposure time (h)'].astype(float)/24
    for i,col in enumerate(['HOMO Energy','LUMO Energy']):
        x[col] = ds['Polymer type'].map(lambda p: ORBITALS[p][i])
    ranges = {}
    for col in FEATURES[2:]:
        lo, hi = float(x[col].min()), float(x[col].max())
        if hi == lo: raise ValueError(f'Constant scaling feature: {col}')
        ranges[col] = {'min':lo,'max':hi}
        x[col] = (x[col]-lo)/(hi-lo)  # Historical leakage deliberately retained.
    y = (ds['Class']=='Negative').astype(int)
    train, test = train_test_split(np.arange(len(ds)), test_size=0.2, random_state=42)
    ds['partition'] = 'train'
    ds.loc[test,'partition'] = 'test'
    assert list(x.columns)==FEATURES and np.isfinite(x.to_numpy()).all()
    overlap = {col: len(set(ds.loc[train,col]) & set(ds.loc[test,col]))
               for col in ['Reference DOI','Particle ID','Article ID']}
    overlap['identical_feature_vectors'] = len(set(map(tuple,x.loc[train].to_numpy())) &
                                             set(map(tuple,x.loc[test].to_numpy())))
    return {'rows':ds,'x':x,'y':y,'train':train,'test':test,
            'scaling':ranges,'overlap':overlap}

"""Metrics for encoded class 1 = CSV Negative, not verified toxic-positive."""
import numpy as np
from sklearn.metrics import roc_auc_score, balanced_accuracy_score, matthews_corrcoef, confusion_matrix

def metrics(y, probability):
    y, p = np.asarray(y).ravel(), np.asarray(probability).ravel()
    if len(y)!=len(p) or not len(y) or not np.isfinite(p).all() or ((p<0)|(p>1)).any():
        raise ValueError('Invalid predictions')
    pred = (p>0.5).astype(int)  # Original neural threshold; RF ties also select class 0.
    two_classes = len(np.unique(y))==2
    cm = confusion_matrix(y,pred,labels=[0,1])
    tn,fp,fn,tp = map(int,cm.ravel())
    # MCC denominator vanishes for constant true labels OR constant predictions.
    mcc_defined = two_classes and len(np.unique(pred))==2
    return {'n':len(y),'class0':int((y==0).sum()),'class1':int((y==1).sum()),
            'roc_auc':float(roc_auc_score(y,p)) if two_classes else np.nan,
            'balanced_accuracy':float(balanced_accuracy_score(y,pred)) if two_classes else np.nan,
            'mcc':float(matthews_corrcoef(y,pred)) if mcc_defined else np.nan,
            'tn':tn,'fp':fp,'fn':fn,'tp':tp,
            'constant_probability':bool(np.ptp(p)==0),
            'collapsed_classes':bool(len(np.unique(pred))==1),
            'metric_note': ('single true class: AUC/BA/MCC undefined' if not two_classes else
                            'constant predicted class: MCC undefined' if not mcc_defined else 'defined')}

"""Fixed CPU training: no tuning, test-based selection, or source retraining."""
import random
import numpy as np
import torch
from torch import nn

def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.set_num_threads(1)
    torch.use_deterministic_algorithms(True)

def fit_network(model, x, y, epochs=50, batch_size=22, lr=0.01, norm_penalty=0.001):
    # For TL this is exactly Adam(model.output_layer.parameters(), lr=0.01).
    params = [p for p in model.parameters() if p.requires_grad]
    optimizer = torch.optim.Adam(params, lr=lr)
    assert {id(p) for g in optimizer.param_groups for p in g['params']} == {id(p) for p in params}
    before = {k:v.clone() for k,v in model.state_dict().items()}
    frozen = {k for k,p in model.named_parameters() if not p.requires_grad}
    x = torch.tensor(np.asarray(x), dtype=torch.float32)
    y = torch.tensor(np.asarray(y), dtype=torch.float32).reshape(-1,1)
    generator = torch.Generator().manual_seed(42)
    history = []
    model.train()
    for epoch in range(epochs):
        order = torch.randperm(len(x), generator=generator)
        total = 0.0
        for indices in order.split(batch_size):
            optimizer.zero_grad()                 # Clear previous gradients.
            probabilities = model(x[indices])      # Forward pass; original sigmoid.
            loss = nn.functional.binary_cross_entropy(probabilities, y[indices])
            # Original sum of tensor L2 norms (NOT squared-weight decay).
            loss = loss + norm_penalty * sum(p.norm(2) for p in model.parameters())
            loss.backward()                       # Compute gradients.
            optimizer.step()                      # Update trainable weights only.
            total += loss.item()*len(indices)
        history.append({'epoch':epoch+1,'loss_with_penalty':total/len(x)})
    after = model.state_dict()
    assert all(torch.equal(before[k], after[k]) for k in frozen)
    changed = [k for k in after if not torch.equal(before[k],after[k])]
    assert any(k.startswith('output_layer.') for k in changed)
    model.eval()
    return history, {'frozen_unchanged':True, 'changed_parameters':changed,
                     'optimizer_only_trainable':True}

def predict(model, x):
    model.eval()
    with torch.no_grad():
        p = model(torch.tensor(np.asarray(x), dtype=torch.float32)).numpy().ravel()
    if not np.isfinite(p).all(): raise ValueError('Nonfinite predictions')
    return p

"""Six reproducible 16:9 figures; saved results are validated before plotting."""
import argparse
import hashlib
import json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib.patches import Circle, Rectangle, Arc
from sklearn.metrics import roc_curve


BLUE, PURPLE, ORANGE, GRAY = '#0072B2','#8B5AA5','#D55E00','#596773'
NAMES = ['step1_initialize_network','step2_load_pretrained_weights',
         'step3_freeze_layers','step4_finetuning','figure5_training_vs_test','figure6_roc_curves']
MODELS = ['Random Forest','Scratch DNN','Transfer Learning']
COLORS = [GRAY, PURPLE, BLUE]

def style():
    plt.rcParams.update({'font.size':16,'axes.titlesize':24,'axes.labelsize':18,
                        'font.family':'DejaVu Sans','svg.fonttype':'none',
                        'figure.facecolor':'white','axes.spines.top':False,
                        'axes.spines.right':False})

def save(fig, folder, name):
    # Refuse overwrites, including generated assets.
    for ext in ['png','svg']:
        p=folder/f'{name}.{ext}'
        if p.exists(): raise FileExistsError(p)
        fig.savefig(p,dpi=300,facecolor='white')
    plt.close(fig)

def lock(ax,x,y):
    ax.add_patch(Rectangle((x-.011,y-.013),.022,.026,facecolor=GRAY))
    ax.add_patch(Arc((x,y+.012),.018,.025,theta1=0,theta2=180,color=GRAY,lw=2))

def network(ax,step, y=.47, scale=1.0, labels=True):
    xs=[.12,.31,.49,.69,.87]
    ns=[4,5,4,3,1]
    nodes=[np.linspace(y-.13*scale,y+.13*scale,n) if n>1 else [y] for n in ns]
    colors=[GRAY,BLUE,BLUE,ORANGE if step==4 else PURPLE,ORANGE if step==4 else PURPLE]
    for i in range(4):
        for a in nodes[i]:
            for b in nodes[i+1]:
                ax.plot([xs[i],xs[i+1]],[a,b],color='#CDD5DE',lw=.8,zorder=0)
    for i in range(5):
        for yy in nodes[i]:
            ax.add_patch(Circle((xs[i],yy),.014*scale,facecolor=colors[i],edgecolor='white',lw=1.5))
        if labels:
            titles=['Inputs\n7 descriptors','Shared 1\n256 + ReLU','Shared 2\n128 + ReLU',
                    'Task-specific hidden\n48 + ReLU','Output\n1 + sigmoid']
            ax.text(xs[i],y-.22,titles[i],ha='center',va='top',fontsize=15)
            if step>=3 and i in [1,2]:
                lock(ax,xs[i],y+.205)
                ax.text(xs[i],y+.245,'Frozen',ha='center',color=GRAY,fontsize=15)
            if step>=3 and i in [3,4]:
                ax.annotate('Updated' if step==4 else 'Trainable',xy=(xs[i],y+.04),
                            xytext=(xs[i],y+.245),ha='center',fontsize=15,color=colors[i],
                            arrowprops={'arrowstyle':'->','color':colors[i],'lw':2})
    return xs

def conceptual(folder):
    style()
    titles=['01  Initialize the neural network','02  Load pretrained nanoparticle weights',
            '03  Freeze the shared layers','04  Fine-tune on MNP observations']
    footers=['Shared feature extraction (blue)  |  Task-specific prediction head (purple)',
             'Reuse learned parameters instead of random initialization.',
             'Preserve pretrained shared layers; update task-specific layers.',
             'Adapt the prediction head using limited MNP data.']
    for step in range(1,5):
        fig,ax=plt.subplots(figsize=(16,9)); fig.subplots_adjust(left=0,right=1,top=1,bottom=0)
        ax.set(xlim=(0,1),ylim=(0,1)); ax.axis('off')
        ax.text(.055,.94,titles[step-1],fontsize=27,weight='bold',va='top')
        network(ax,step)
        if step==1:
            ax.text(.5,.79,'7 → 256 → 128 → 48 → 1',ha='center',fontsize=24,color=GRAY)
        if step==2:
            ax.text(.5,.84,'Pretrained nanoparticle DNN  →  transfer all 8 parameter tensors',ha='center',fontsize=20)
            network(ax,2,y=.755,scale=.22,labels=False)
            for x in [.31,.49,.69,.87]:
                ax.annotate('',xy=(x,.63),xytext=(x,.71),arrowprops={'arrowstyle':'->','lw':2,'color':ORANGE})
            ax.text(.12,.66,'MNP DNN',ha='center',fontsize=14)
        if step==4:
            ax.annotate('80 MNP training\nobservations',xy=(.12,.59),xytext=(.12,.78),
                        ha='center',fontsize=16,arrowprops={'arrowstyle':'->','color':GRAY,'lw':2})
        ax.text(.5,.13,footers[step-1],ha='center',fontsize=21)
        ax.text(.5,.065,'Representative neurons; actual dimensions labeled. Output: P(encoded class 1 = Negative).',
                ha='center',fontsize=13,color=GRAY)
        save(fig,folder,NAMES[step-1])

def validated_results(results):
    manifest=json.loads((results/'result_hashes.json').read_text())
    for name,digest in manifest.items():
        if hashlib.sha256((results/name).read_bytes()).hexdigest()!=digest:
            raise ValueError(f'Result integrity check failed: {name}')
    perf=pd.read_csv(results/'performance.csv')
    pred=pd.read_csv(results/'predictions.csv')
    for _,row in perf.iterrows():
        sub=pred[(pred.model==row.model)&(pred.partition==row.partition)]
        calculated=metrics(sub.y_true,sub.p_class1)
        for key in ['n','class0','class1','roc_auc','balanced_accuracy','mcc','tn','fp','fn','tp']:
            if not np.isclose(calculated[key],row[key],atol=1e-12,equal_nan=True):
                raise ValueError(f'Metric mismatch: {row.model}/{row.partition}/{key}')
    return perf,pred

def results_figures(results,folder):
    style(); perf,pred=validated_results(results)
    fig,ax=plt.subplots(figsize=(16,9)); fig.subplots_adjust(bottom=.22,top=.86,left=.10,right=.96)
    for j,(partition,color) in enumerate([('train',PURPLE),('test',BLUE)]):
        vals=[perf[(perf.model==m)&(perf.partition==partition)].roc_auc.iloc[0] for m in MODELS]
        bars=ax.bar(np.arange(3)+(j-.5)*.34,vals,.32,label=partition.title(),color=color)
        for b,v in zip(bars,vals):
            ax.text(b.get_x()+b.get_width()/2,v+.018,f'{v:.3f}',ha='center',fontsize=18)
    ax.set(xticks=np.arange(3),xticklabels=MODELS,ylim=(0,1.13),ylabel='ROC-AUC',
           title='05  Model fitting versus held-out performance')
    ax.legend(loc='upper center', bbox_to_anchor=(.5, 1.0), ncol=2); ax.grid(axis='y',alpha=.15); ax.set_axisbelow(True)
    fig.text(.5,.09,'N = 100 observations; train = 80, test = 20. Historical observation-level split.',ha='center',fontsize=16)
    fig.text(.5,.045,'Training performance alone does not establish predictive utility; study overlap and scaling leakage remain.',ha='center',fontsize=14)
    save(fig,folder,NAMES[4])
    fig,ax=plt.subplots(figsize=(16,9)); fig.subplots_adjust(bottom=.20,top=.87,left=.11,right=.95)
    for m,c in zip(MODELS,COLORS):
        sub=pred[(pred.model==m)&(pred.partition=='test')]
        if sub.y_true.nunique()==2:
            fpr,tpr,_=roc_curve(sub.y_true,sub.p_class1)
            a=metrics(sub.y_true,sub.p_class1)['roc_auc']
            ax.plot(fpr,tpr,lw=3,color=c,label=f'{m}: AUC = {a:.3f}')
        else: ax.plot([],[],color=c,label=f'{m}: AUC undefined')
    ax.plot([0,1],[0,1],'--',color='#999999',label='Chance reference')
    ax.set(xlim=(0,1),ylim=(0,1.03),xlabel='False positive rate',ylabel='True positive rate',
           title='06  Held-out ROC curves | same 20 observations')
    ax.legend(loc='lower right'); ax.grid(alpha=.15)
    fig.text(.5,.065,'Positive metric class = encoded 1 (CSV Negative). Single split; no uncertainty intervals.',ha='center',fontsize=16)
    save(fig,folder,NAMES[5])
    # Exact numerical inputs to the scientific figures, independently checkable.
    perf[['model','partition','roc_auc']].to_csv(folder/'figure5_values.csv',index=False)
    points=[]
    for m in MODELS:
        sub=pred[(pred.model==m)&(pred.partition=='test')]
        if sub.y_true.nunique()==2:
            fpr,tpr,threshold=roc_curve(sub.y_true,sub.p_class1)
            points.extend({'model':m,'fpr':f,'tpr':t,'threshold':h} for f,t,h in zip(fpr,tpr,threshold))
    pd.DataFrame(points).to_csv(folder/'figure6_values.csv',index=False)

def generate(results,folder):
    folder=Path(folder); folder.mkdir(parents=True,exist_ok=False)
    validated_results(Path(results))
    conceptual(folder); results_figures(Path(results),folder)



### 0.3 Download and Prepare the MNP Dataset

The dataset is downloaded automatically and verified by SHA-256. The original published preprocessing is preserved for educational reproduction, including normalization before splitting and the documented observation-level split.

The output below reports **100 total observations**, **80 training observations**, **20 testing observations**, and the class distributions. Class 0 means `Positive`; class 1 means `Negative` (biological toxicity semantics remain unresolved). These are observations, not unique particles.

In [ ]:
dataset_path = download_verified('Datasets/MNP dataset.csv', RUN / 'mnp_data.csv',
    '13c1717103a82b07f9201fd3cafe14ac9ec3eef15d42a2f0684d2d9c68a3cd49')
data = prepare_data(dataset_path)
x, y, train, test = data['x'], data['y'], data['train'], data['test']
seed_everything(42)
conceptual(FIG)
print('Seven inputs in order:', FEATURES)
print('N = 100 total observations, not unique particles')
print('Train:', len(train), y.loc[train].value_counts().to_dict())
print('Test:', len(test), y.loc[test].value_counts().to_dict())

## Part 1. Hands-On Transfer Learning Tutorial

The following five steps correspond directly to the SOT CE presentation slides. Each step includes executable Python code, a conceptual figure, and the expected output.

### Slide 1 — Initialize the Neural Network
Shared representation layers transform seven descriptors. A **48-neuron task-specific hidden layer** feeds the one-neuron sigmoid output: 7 → 256 → 128 → 48 → 1.

In [ ]:
class DNN(nn.Module):
    def __init__(self, input_size=7, shared_layer_sizes=(256, 128),
                 output_layer_size=1, specific_layer_size=48):
        super().__init__()
        widths = [input_size, *shared_layer_sizes]
        self.shared_layers = nn.ModuleList([
            nn.Linear(a, b) for a, b in zip(widths[:-1], widths[1:])])
        self.output_layer = nn.Sequential(
            nn.Linear(widths[-1], specific_layer_size), nn.ReLU(),
            nn.Linear(specific_layer_size, output_layer_size))

    def forward(self, x):
        for layer in self.shared_layers:
            x = torch.relu(layer(x))
        return torch.sigmoid(self.output_layer(x))

model = DNN(input_size=7, shared_layer_sizes=[256, 128],
            output_layer_size=1, specific_layer_size=48)
assert model(torch.zeros(5, 7)).shape == (5, 1)
print(model)
display(Image(filename=str(FIG / 'step1_initialize_network.png'), width=1000))

**Expected observation:** a randomly initialized network with two shared layers, a 48-unit hidden head layer, and one probability output.

### Slide 2 — Download and Load Pretrained Weights
Reuse all pretrained parameter tensors, including the shared layers **and prediction head**. A frozen Git commit and checksum identify the original model; tensor compatibility does not establish equivalent source/target descriptor meanings.

In [ ]:
checkpoint_path = download_verified('Code/best_pre-trained_model.pth',
    RUN / 'best_pre-trained_model.pth',
    'd046253431a9681219385c3f4bd9eb7339d4f51c1413941ba234ab3aaab36241')
state_dict = torch.load(checkpoint_path, map_location='cpu', weights_only=True)
expected = model.state_dict()
assert set(state_dict) == set(expected), 'Parameter names must match'
for name, tensor in state_dict.items():
    assert tensor.shape == expected[name].shape, name
    assert torch.isfinite(tensor).all(), name
assert state_dict['shared_layers.0.weight'].shape[1] == 7
model.load_state_dict(state_dict, strict=True)
assert all(torch.equal(model.state_dict()[k], v) for k, v in state_dict.items())
print('Loaded all', len(state_dict), 'parameter tensors strictly.')
display(Image(filename=str(FIG / 'step2_load_pretrained_weights.png'), width=1000))

**Expected observation:** all eight weight/bias tensors load successfully after checksum verification. No source model is retrained.

### Slide 3 — Freeze Shared Layers
Preserve the pretrained shared representations. Only the task-specific head will adapt to the MNP training observations.

In [ ]:
for param in model.shared_layers.parameters():
    param.requires_grad = False
optimizer = torch.optim.Adam(model.output_layer.parameters(), lr=0.01)
frozen = sum(p.numel() for p in model.parameters() if not p.requires_grad)
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
assert (frozen, trainable) == (34944, 6241)
assert all(not p.requires_grad for p in model.shared_layers.parameters())
assert all(p.requires_grad for p in model.output_layer.parameters())
assert {id(p) for group in optimizer.param_groups for p in group['params']} == {
    id(p) for p in model.parameters() if p.requires_grad}
print(f'Frozen: {frozen:,}; trainable: {trainable:,}')
display(Image(filename=str(FIG / 'step3_freeze_layers.png'), width=1000))

**Expected observation:** 34,944 parameters are frozen and 6,241 remain trainable; the optimizer contains only the head.

### Slide 4 — Fine-Tune on MNP Data
Adapt the head with 80 training observations for 50 epochs, batches of 22, Adam learning rate 0.01. Keep the original sigmoid and binary cross-entropy; the regularizer is a sum of L2 norms, not squared-weight decay.

In [ ]:
before = {k: v.clone() for k, v in model.state_dict().items()}
x_train = torch.tensor(x.loc[train].to_numpy(), dtype=torch.float32)
y_train = torch.tensor(y.loc[train].to_numpy(), dtype=torch.float32).reshape(-1, 1)
generator = torch.Generator().manual_seed(42)
model.train()
for epoch in range(50):
    order = torch.randperm(len(x_train), generator=generator)
    for indices in order.split(22):
        x_batch, y_batch = x_train[indices], y_train[indices]
        optimizer.zero_grad()  # Clear previous gradients.
        probability = model(x_batch)  # Forward pass.
        loss = torch.nn.functional.binary_cross_entropy(probability, y_batch)
        loss = loss + 0.001 * sum(p.norm(2) for p in model.parameters())
        loss.backward()  # Compute gradients.
        optimizer.step()  # Update only trainable parameters.
model.eval()
after = model.state_dict()
assert all(torch.equal(before[k], after[k]) for k in before if k.startswith('shared_layers.'))
head_changed = [k for k in before if k.startswith('output_layer.') and not torch.equal(before[k], after[k])]
assert len(head_changed) == 4
print('Shared weights unchanged; updated:', head_changed)
display(Image(filename=str(FIG / 'step4_finetuning.png'), width=1000))

**Expected observation:** shared tensors are exactly unchanged, while all four task-specific weight/bias tensors update. Test observations never enter optimizer updates.

### Slide 5 — Evaluate Predictive Performance
Compare RF, scratch DNN, and transfer learning on the same 80/20 split. RF and scratch are target-only baselines; all metrics and curves below are calculated from actual predictions, not copied from prior results.

In [ ]:
from sklearn.ensemble import RandomForestClassifier
seed_everything(42)
scratch = DNN(7, [256, 128], 1, 48)
fit_network(scratch, x.loc[train], y.loc[train])
rf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=1)
rf.fit(x.loc[train], y.loc[train])
models = {'Transfer Learning': model, 'Scratch DNN': scratch, 'Random Forest': rf}
metric_rows, prediction_frames = [], []
for name, fitted in models.items():
    for partition, indices in [('train', train), ('test', test)]:
        probability = (fitted.predict_proba(x.loc[indices])[:, 1] if name == 'Random Forest'
                       else predict(fitted, x.loc[indices]))
        metric_rows.append({'model': name, 'partition': partition, **metrics(y.loc[indices], probability)})
        prediction_frames.append(pd.DataFrame({'model': name, 'partition': partition,
            'sample_row': indices, 'source_row': data['rows'].loc[indices, '_source_row'].to_numpy(),
            'y_true': y.loc[indices].to_numpy(), 'p_class1': probability,
            'prediction': (probability > 0.5).astype(int)}))
performance = pd.DataFrame(metric_rows)
predictions = pd.concat(prediction_frames)
performance.to_csv(RUN / 'performance.csv', index=False)
predictions.to_csv(RUN / 'predictions.csv', index=False)
manifest = {name: hashlib.sha256((RUN / name).read_bytes()).hexdigest()
            for name in ['performance.csv', 'predictions.csv']}
(RUN / 'result_hashes.json').write_text(json.dumps(manifest, indent=2))
display(performance)
results_figures(RUN, FIG)
for name in ['figure5_training_vs_test', 'figure6_roc_curves']:
    display(Image(filename=str(FIG / (name + '.png')), width=1000))
environment = {'python': platform.python_version(), 'seed':42, 'device':'cpu',
               'packages': {p:importlib.metadata.version(p) for p in required},
               'overlap':data['overlap']}
(RUN / 'environment.json').write_text(json.dumps(environment, indent=2))
print('Versions:', environment['packages'])
print('Outputs are saved in a new colab_runs folder.')

**Expected observation:** six train/test metric rows and two result figures. The previous educational run had RF/scratch/TL test AUC ≈ 0.800/0.773/0.867; this notebook recomputes them. Balanced accuracy and MCC use p>0.5, with confusion matrix rows=true [0,1], columns=predicted [0,1]. Undefined metrics remain missing; constant probabilities and collapsed classifications are flagged.

Training performance alone does not establish predictive utility; a train–test gap alone is not definitive evidence of overfitting. No uncertainty intervals are inferred from this single small split.

## Scientific limitations and attribution
- Source and target descriptor positions do not all represent the same physical quantities.
- Target class 1 is `Negative`; a definitive toxic/non-toxic meaning is unverified.
- Historical min-max normalization uses all 100 sampled observations before splitting.
- Train/test share studies, particle IDs and some feature vectors; this is not independent-study validation.
- Simplified full-pool training differs from the published fold-selection procedure; historical N=50/100/200 results are not generated here.
- These results teach transfer mechanics and do not resolve external-generalization limitations.

See [scientific notes](https://github.com/choulab210/SOT-2027-CE-Course/blob/main/SCIENTIFIC_NOTES.md) and [provenance/rights](https://github.com/choulab210/SOT-2027-CE-Course/blob/main/THIRD_PARTY_PROVENANCE.md). Cite Chen et al. (2025), *JHM* 496, 139353, [DOI](https://doi.org/10.1016/j.jhazmat.2025.139353), and the [original repository](https://github.com/choulab210/Micro-Nanoplastics).

Original notebook/code contributions: MIT. Original standalone educational figures and documentation: CC BY 4.0. Original datasets, weights and third-party materials retain their own rights and are not relicensed.